# 03 — Native YOLO26n Training

Inspect the completed historical detector or run native Ultralytics on explicitly prepared offline S20 data. Raw development remains the common checkpoint-selection input.

**Inputs:** Experiment YAML; runs/v1/data/data.yaml and saved v0 Raw development export for new training; historical CSV for replay.

**Outputs:** runs/v1/train/{args.yaml,results.csv,weights/} only after explicit opt-in.

**Prerequisites:** Run the preparation phase separately; pretrained YOLO26n checkpoint is local. No upstream notebook is executed.

**Outline**

- [1. Configuration](#section-1)
- [2. Native Training or Saved History](#section-2)
- [3. Training Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

Historical S10 results are the default; active exports select S20. New training requires both switches; native resume is explicit.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from vehicle_detection.common import require_paths

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
from vehicle_detection.experiment_data import load_experiment

EXPERIMENT = "v1"
require_paths(config=ROOT / f"configs/{EXPERIMENT}.yaml", shared=ROOT / "configs/data.yaml")
config = load_experiment(ROOT, EXPERIMENT)
from vehicle_detection.finetuning import train_experiment

READ_HISTORICAL = True
ALLOW_TRAINING = False
RESUME = False
HISTORICAL_CSV = ROOT / "docs/issue21/adaptive_results.csv"

<a id="section-2"></a>

## 2. Native Training or Saved History

Both detectors start independently from the pinned COCO checkpoint. Native Ultralytics owns optimization, history and resume.

In [ ]:
if READ_HISTORICAL:
    require_paths(history=HISTORICAL_CSV)
    history = pd.read_csv(HISTORICAL_CSV)
    print("Historical measurement; no new training occurred.")
else:
    history = train_experiment(ROOT, EXPERIMENT, allow_training=ALLOW_TRAINING, resume=RESUME)
display(history[["epoch", "metrics/mAP50-95(B)", "train/box_loss"]])

<a id="section-3"></a>

## 3. Training Summary

Historical Raw resumed after epoch 12; native resume is not bitwise uninterrupted training. No training-seed uncertainty or superiority claim follows from these histories.

In [ ]:
print("Epochs visible:", len(history))
print("Historical native artifacts: runs/training-five-s10-v1/")
print("New native artifacts:", ROOT / f"runs/{EXPERIMENT}/train")
print("Final holdout stays sealed.")